In [ ]:
from ui.gradio_app import HealthcareChatbotApp

if __name__ == "__main__":

    HealthcareChatbotApp().create_ui().launch()

기존 FAISS 인덱스 로드 (데이터 변경 없음)
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [2]:
import gradio as gr

from service.chat_service import ChatService

class HealthcareChatbotApp:

    def __init__(self):
        self.chat_service = ChatService()

    def create_wrapper(self, message, history):
        if not message.strip():
            return "질문내용을 남겨주세요."
        
        return self.chat_service.get_response(message, history)
    
    def create_ui(self):
        return gr.ChatInterface(
            fn=self.create_wrapper,
            title="디지털 헬스케어 챗봇",
            description="내과 관련 질문을 해보세요!",
            examples=[
                "급성 위장염은 어떻게 치료하나요?",
                "당뇨병 환자의 식이요법이 궁금해요",
                "고혈압은 어떻게 관리하나요?",
                "갑상선암 초기 증상은 어떻게 되나요?",
            ]
        )


In [3]:
HealthcareChatbotApp().create_ui().launch()

   ↳ 전체 2,212,306개 중 진료과='내과' 에서 질병 264종 10,000건 적재 (질병당 최대 50, 전체 최대 10000)
✅ FAQ 데이터 로드 완료: 10000개 QA (출처: aihub/answer_files)
기존 FAISS 인덱스 로드 (데이터 변경 없음)
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

from __future__ import annotations

@dataclass
class FAQItems:
    id:str
    category:str 
    question:str
    answer:str
    keywords:list[str] = field(default_factory=list)
    difficulty:str = "unknown"
    source:str = "unknown"

def to_dict(self)-> dict[str, Any]:
    return asdict(self)
    
def _first_present(record: dict, keys:list[str]) -> Optional[Any]:

    for k in keys:
            if k in record and record[k] is not None:
                return record[k]
    return None
        
def normalize_record(record:dict, field_map:dict, index:int, source:str) -> Optional[FAQItems]:

    if not isinstance(record, dict):
        return None
    
    keywords = record.get("keywords") or record.get("키워드")

    if isinstance(keywords, str):
        keywords = [keywords]

    question = _first_present(record, field_map["question_keys"])
    answer = _first_present(record, field_map["answer_keys"])

    if not question or not answer:
        return None
    
    category = _first_present(record, field_map["category_keys"]) or "미분류"
    raw_id = _first_present(record, field_map["id_keys"])
    item_id = str(raw_id) if raw_id is not None else f"{source}-{index:06d}"

    return FAQItems(
        id=item_id,
        category=str(category).strip(),
        question=str(question).strip(),
        answer=str(answer).strip(),
        keywords=list(keywords),
        difficulty=str(record.get("difficulty", "unknown")),
        source=str(source).strip(),
    )





